# V12 — Truong (2018) Spectrogram-CNN under Cross-Patient LOPO

**Direct literature copy.** This notebook implements the most-cited
seizure-prediction methodology — Truong et al. (2018), *"Convolutional
neural networks for seizure prediction using intracranial and scalp
electroencephalogram"*, Neural Networks 105 — and applies it under strict
cross-patient LOPO instead of the patient-specific protocol used in the
original paper.

**Replicated from Truong (2018):**
- Short-Time Fourier Transform (STFT) on each EEG window
- 1-second STFT window (nperseg = FS = 256), 50 % overlap (noverlap = 128)
- Log-magnitude spectrogram (`log(1 + |STFT|²)`) over 0.5–30 Hz
- Per-window input tensor: (18 channels, 31 frequency bins, 39 time bins)
- 3-block convolutional architecture: Conv2D → BatchNorm → ReLU → MaxPool
- FC head: 256-unit Linear → Dropout → 1-unit Sigmoid
- Binary cross-entropy with class-frequency reweighting
- Adam optimiser (lr = 1e-3), early stopping on validation loss

**Changed from Truong (2018):**
- Cross-patient LOPO instead of patient-specific evaluation (this is the
  whole point of the experiment — to test cross-patient transfer of their
  features and architecture)
- Same alarm-level post-processing pipeline (smoothing → operational
  threshold → K-of-M vote + refractory) used throughout this thesis

**Why this matters for the thesis.** Truong's methodology achieves
Sens = 0.81 @ FPR/h = 0.16 under patient-specific evaluation. Applying it
under LOPO directly tests whether the published gain is attributable to
the spectrogram-CNN methodology or to the patient-specific protocol. The
answer feeds RQ2 (cross-patient generalisation) with literature-grounded
empirical evidence.


In [1]:
# --- portable repo bootstrap (added for public release; locates the repo root) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
CODE_DIR = str(REPO); CODE = REPO; CODEV2 = REPO; PROJECT_DIR = REPO
# --------------------------------------------------------------------------------

# Cell 0 — Imports & config
import os, sys, json, copy, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.signal import spectrogram

# [path set by bootstrap] CODE_DIR = r"<repo>/Code"
sys.path.insert(0, CODE_DIR)
from config import (
    DATA_ROOT, CANONICAL_CHANNELS, N_CHANNELS, FS, STEP_SEC,
    EXCLUDED_PATIENTS, RESULTS_DIR, RANDOM_SEED,
    INTERICTAL_MULTIPLIER, MAX_INTERICTAL_ABS,
    ALARM_K, ALARM_M, ALARM_REFRACTORY,
)
from summary_parser import parse_all_summaries
from data_loader   import load_edf
from preprocessing import bandpass_filter, segment_signal, label_windows
from metrics       import evaluate_with_alarms
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

import importlib, alarm_postproc
importlib.reload(alarm_postproc)
from alarm_postproc import smooth_probs, operational_threshold

torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Truong (2018) STFT parameters, adapted to FS=256
TRUONG_NPERSEG  = 256        # 1-second STFT window
TRUONG_NOVERLAP = 128        # 50 % overlap
FREQ_LO, FREQ_HI = 0.5, 30.0 # 0.5–30 Hz band (matches our bandpass)

SMOOTH_WINDOW = 10
TARGET_FPR_H  = 1.0

all_seizures = parse_all_summaries(DATA_ROOT)
patients = sorted([p for p in os.listdir(DATA_ROOT)
                   if os.path.isdir(os.path.join(DATA_ROOT, p))
                   and p.startswith('chb') and p not in EXCLUDED_PATIENTS])
print(f'Device: {DEVICE}  Patients: {len(patients)}')
print(f'STFT: nperseg={TRUONG_NPERSEG}, noverlap={TRUONG_NOVERLAP}, band={FREQ_LO}-{FREQ_HI}Hz')


[INFO] chb01: 7 seizure-containing files, 7 total seizures
[INFO] chb02: 3 seizure-containing files, 3 total seizures
[INFO] chb03: 7 seizure-containing files, 7 total seizures
[INFO] chb04: 3 seizure-containing files, 4 total seizures
[INFO] chb05: 5 seizure-containing files, 5 total seizures
[INFO] chb06: 7 seizure-containing files, 10 total seizures
[INFO] chb07: 3 seizure-containing files, 3 total seizures
[INFO] chb08: 5 seizure-containing files, 5 total seizures
[INFO] chb09: 3 seizure-containing files, 4 total seizures
[INFO] chb10: 7 seizure-containing files, 7 total seizures
[INFO] chb11: 3 seizure-containing files, 3 total seizures
[INFO] chb12: 13 seizure-containing files, 40 total seizures
[INFO] chb13: 8 seizure-containing files, 12 total seizures
[INFO] chb14: 7 seizure-containing files, 8 total seizures
[INFO] chb15: 14 seizure-containing files, 20 total seizures
[INFO] chb16: 6 seizure-containing files, 10 total seizures
[INFO] chb17: 3 seizure-containing files, 3 total

In [2]:
# Cell 1 — Truong-style log-spectrogram feature extraction
def truong_spectrogram(window, fs=FS):
    """
    Compute the Truong (2018) log-spectrogram representation for one EEG
    window of shape (n_channels, n_samples).

    Returns a tensor (n_channels, n_freqs_cropped, n_time_bins) of float32.
    """
    f, t, Sxx = spectrogram(
        window, fs=fs,
        nperseg=TRUONG_NPERSEG,
        noverlap=TRUONG_NOVERLAP,
        scaling='density',
        mode='psd',
    )
    band = (f >= FREQ_LO) & (f <= FREQ_HI)
    Sxx  = Sxx[:, band, :]            # crop to 0.5–30 Hz
    return np.log1p(Sxx).astype(np.float32)

# Probe an output shape on a dummy window
_dummy = np.random.randn(N_CHANNELS, 20 * FS).astype(np.float32)
_S = truong_spectrogram(_dummy)
print(f'Truong spectrogram per window: shape={_S.shape}, '
      f'dtype={_S.dtype}, size={_S.nbytes/1024:.1f} KB')
SPEC_SHAPE = _S.shape  # (n_channels, n_freqs, n_time)


Truong spectrogram per window: shape=(18, 30, 39), dtype=float32, size=82.3 KB


In [3]:
# Cell 2 — Build per-patient Truong-spectrogram dataset (one EDF pass)
def build_spectrograms():
    data = {}
    for pid in patients:
        fmap = all_seizures.get(pid, {})
        if not fmap: continue
        print(f'\nPatient {pid}')
        Xs, ys = [], []
        for fname in sorted(fmap):
            seizures = fmap[fname]
            if not seizures: continue
            edf_path = os.path.join(DATA_ROOT, pid, fname)
            if not os.path.exists(edf_path): continue
            try:
                signal, fs = load_edf(edf_path)
            except Exception:
                continue
            filtered = bandpass_filter(signal)
            windows  = segment_signal(filtered)
            labels   = label_windows(filtered.shape[1], seizures)
            valid    = labels != -1
            windows  = windows[valid]; labels = labels[valid]
            if (labels == 1).sum() == 0: continue
            Xs.append(np.stack([truong_spectrogram(w) for w in windows]))
            ys.append(labels)
            print(f'  {fname}: pre={(labels==1).sum()} int={(labels==0).sum()}')
        if not ys: continue
        X = np.concatenate(Xs); y = np.concatenate(ys)
        # Apply same per-patient interictal cap as every other notebook
        n_pre, n_int = int((y==1).sum()), int((y==0).sum())
        cap = min(n_int, INTERICTAL_MULTIPLIER * n_pre, MAX_INTERICTAL_ABS)
        if n_int > cap:
            rng = np.random.default_rng(RANDOM_SEED + hash(pid) % 10_000)
            int_idx = np.where(y == 0)[0]
            keep_int = rng.choice(int_idx, size=cap, replace=False)
            pre_idx  = np.where(y == 1)[0]
            keep = np.sort(np.concatenate([pre_idx, keep_int]))
            X, y = X[keep], y[keep]
        data[pid] = (X.astype(np.float32), y.astype(np.int8))
    return data

t0 = time.time()
spec_data = build_spectrograms()
print(f'\nSpectrogram extraction done in {(time.time()-t0)/60:.1f}min')
patient_ids = sorted(spec_data.keys())
print(f'Loaded {len(patient_ids)} patients')

total_size_mb = sum(X.nbytes for X, _ in spec_data.values()) / 1024**2
print(f'Total memory footprint: {total_size_mb:.0f} MB')



Patient chb01
  chb01_03.edf: pre=148 int=146
    [LABEL] Seizure at 1467s: preictal window out of bounds (would start at -333s) — skipping preictal label.
    [LABEL] Seizure at 1732s: preictal window out of bounds (would start at -68s) — skipping preictal label.
    [LABEL] Seizure at 1015s: preictal window out of bounds (would start at -785s) — skipping preictal label.
    [LABEL] Seizure at 1720s: preictal window out of bounds (would start at -80s) — skipping preictal label.
    [LABEL] Seizure at 327s: preictal window out of bounds (would start at -1473s) — skipping preictal label.
  chb01_26.edf: pre=148 int=33

Patient chb02
  chb02_16+.edf: pre=148 int=144
    [LABEL] Seizure at 130s: preictal window out of bounds (would start at -1670s) — skipping preictal label.
  chb02_19.edf: pre=148 int=183

Patient chb03
    [LABEL] Seizure at 362s: preictal window out of bounds (would start at -1438s) — skipping preictal label.
    [LABEL] Seizure at 731s: preictal window out of bounds 

In [4]:
# Cell 3 — Truong-style CNN (3 conv blocks → FC head)
class TruongCNN(nn.Module):
    def __init__(self, in_channels=N_CHANNELS, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, *SPEC_SHAPE)
            flat  = self.features(dummy).numel()
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(flat, 256), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(256, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


class SpecDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


def train_truong(model, X_tr, y_tr, X_va, y_va, epochs=15, patience=5, lr=1e-3, bs=64):
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    pos_w = (len(y_tr) - (y_tr == 1).sum()) / max((y_tr == 1).sum(), 1)
    bce   = nn.BCELoss(reduction='none')
    ld_tr = DataLoader(SpecDataset(X_tr, y_tr), batch_size=bs, shuffle=True, drop_last=True)
    best, best_state, patience_ct = float('inf'), None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in ld_tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            p = model(xb).clamp(1e-7, 1 - 1e-7)
            w = torch.where(yb == 1, torch.full_like(yb, float(pos_w)), torch.ones_like(yb))
            loss = (w * bce(p, yb)).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            xv = torch.tensor(X_va, dtype=torch.float32).to(DEVICE)
            yv = torch.tensor(y_va, dtype=torch.float32).to(DEVICE)
            pv = model(xv).clamp(1e-7, 1 - 1e-7)
            vl = bce(pv, yv).mean().item()
        if vl < best:
            best = vl; best_state = copy.deepcopy(model.state_dict()); patience_ct = 0
        else:
            patience_ct += 1
            if patience_ct >= patience: break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict_truong(model, X, bs=128):
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = torch.tensor(X[i:i+bs], dtype=torch.float32).to(DEVICE)
        out.append(model(xb).cpu().numpy())
    return np.concatenate(out)

print('Truong CNN architecture and trainer ready.')
print(f'Expected feature-map flat size: deduced from SPEC_SHAPE={SPEC_SHAPE}')


Truong CNN architecture and trainer ready.
Expected feature-map flat size: deduced from SPEC_SHAPE=(18, 30, 39)


In [5]:
# Cell 4 — Cross-patient LOPO with smoothing + alarm-level operational threshold
from seizure_metrics import generate_alarms, false_alarms_per_hour
from config import ALARM_K, ALARM_M, ALARM_REFRACTORY, STEP_SEC

def run_truong_lopo(data, max_train_per_fold=20_000):
    rows_alarm, rows_cmp = [], []
    pids = sorted(data.keys())
    print(f'\n== Truong CNN cross-patient LOPO ({len(pids)} folds, '
          f'smooth={SMOOTH_WINDOW}w, alarm-level FPR/h ≤ {TARGET_FPR_H}) ==')
    t0 = time.time()
    for fi, test_pid in enumerate(pids, 1):
        Xs = [data[p][0] for p in pids if p != test_pid]
        ys = [data[p][1] for p in pids if p != test_pid]
        X_tr = np.concatenate(Xs); y_tr = np.concatenate(ys)
        X_te, y_te = data[test_pid]

        if max_train_per_fold and len(X_tr) > max_train_per_fold:
            rng = np.random.default_rng(RANDOM_SEED + fi)
            idx = rng.choice(len(X_tr), size=max_train_per_fold, replace=False)
            X_tr, y_tr = X_tr[idx], y_tr[idx]

        tr_i, va_i = train_test_split(np.arange(len(y_tr)), test_size=0.15,
                                      random_state=RANDOM_SEED, stratify=y_tr)
        X_va, y_va = X_tr[va_i], y_tr[va_i]
        X_tr, y_tr = X_tr[tr_i], y_tr[tr_i]

        model = TruongCNN().to(DEVICE)
        model = train_truong(model, X_tr, y_tr, X_va, y_va)
        probs_raw = predict_truong(model, X_te)

        if len(np.unique(y_te)) < 2: continue

        probs    = smooth_probs(probs_raw, window=SMOOTH_WINDOW)
        threshold = operational_threshold(
            probs, y_te, STEP_SEC, target_fpr_h=TARGET_FPR_H,
            K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
        )

        a = evaluate_with_alarms(
            y_true=y_te, probs=probs, threshold=threshold,
            K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
            patient_id=test_pid,
        )
        a['model'] = 'TruongCNN'; a['threshold'] = threshold
        rows_alarm.append(a)

        fpr_a, tpr_a, thr = roc_curve(y_te, probs_raw)
        thr_raw = float(np.clip(thr[np.argmax(tpr_a - fpr_a)], 0.05, 0.95))
        pred_w = (probs_raw >= thr_raw).astype(int)
        tp=int(((pred_w==1)&(y_te==1)).sum()); fp=int(((pred_w==1)&(y_te==0)).sum())
        tn=int(((pred_w==0)&(y_te==0)).sum()); fn=int(((pred_w==0)&(y_te==1)).sum())
        sens_w = tp/max(tp+fn,1)
        hours_int = (y_te==0).sum() * STEP_SEC / 3600.0
        fpr_w = false_alarms_per_hour(generate_alarms(pred_w.astype(float), 0.5, ALARM_K, ALARM_M, ALARM_REFRACTORY), y_te, STEP_SEC)
        rows_cmp.append(dict(patient=test_pid, model='TruongCNN',
            auc=a['auc'], auc_pr=a['auc_pr'], threshold=threshold,
            sens_window=sens_w, fpr_h_window=fpr_w,
            sens_alarm=a['sensitivity'], fpr_h_alarm=a['fpr_per_hour']))
        print(f'  [{fi:2d}/{len(pids)}] {test_pid}  AUC={a["auc"]:.3f}  '
              f'AUC-PR={a["auc_pr"]:.3f}  thr={threshold:.2f}  '
              f'Sens(a)={a["sensitivity"]:.3f}  FPR/h(a)={a["fpr_per_hour"]:.2f}')
    print(f'  Done in {(time.time()-t0)/60:.1f}min')
    return rows_alarm, rows_cmp

rows_alarm, rows_cmp = run_truong_lopo(spec_data)



== Truong CNN cross-patient LOPO (21 folds, smooth=10w, alarm-level FPR/h ≤ 1.0) ==
  [ 1/21] chb01  AUC=0.527  AUC-PR=0.636  thr=0.47  Sens(a)=0.000  FPR/h(a)=0.00
  [ 2/21] chb02  AUC=0.505  AUC-PR=0.500  thr=0.44  Sens(a)=0.000  FPR/h(a)=0.00
  [ 3/21] chb03  AUC=0.524  AUC-PR=0.695  thr=0.31  Sens(a)=0.000  FPR/h(a)=0.00
  [ 4/21] chb04  AUC=0.483  AUC-PR=0.162  thr=0.48  Sens(a)=0.000  FPR/h(a)=0.00
  [ 5/21] chb05  AUC=0.518  AUC-PR=0.647  thr=0.47  Sens(a)=0.000  FPR/h(a)=0.00
  [ 6/21] chb06  AUC=0.452  AUC-PR=0.177  thr=0.47  Sens(a)=0.000  FPR/h(a)=0.00
  [ 7/21] chb07  AUC=0.502  AUC-PR=0.167  thr=0.49  Sens(a)=0.000  FPR/h(a)=0.00
  [ 8/21] chb08  AUC=0.528  AUC-PR=0.595  thr=0.35  Sens(a)=0.000  FPR/h(a)=0.00
  [ 9/21] chb09  AUC=0.561  AUC-PR=0.236  thr=0.41  Sens(a)=0.000  FPR/h(a)=0.00
  [10/21] chb10  AUC=0.519  AUC-PR=0.267  thr=0.45  Sens(a)=0.000  FPR/h(a)=0.00
  [11/21] chb13  AUC=0.454  AUC-PR=0.520  thr=0.42  Sens(a)=0.000  FPR/h(a)=0.00
  [12/21] chb14  AUC=0.5

In [6]:
# Cell 5 — Save and summarise
df_alarm = pd.DataFrame(rows_alarm)
df_cmp   = pd.DataFrame(rows_cmp)
df_alarm.to_csv(Path(RESULTS_DIR) / 'lopo_v12_truong_alarm.csv', index=False)
df_cmp.to_csv(Path(RESULTS_DIR)   / 'lopo_v12_truong_compare.csv', index=False)

print(f'{"Model":<11} {"AUC":>6} {"AUC-PR":>7} '
      f'{"Sens(w)":>8} {"FPR/h(w)":>10} {"Sens(a)":>8} {"FPR/h(a)":>10}')
print('-' * 70)
print(f'{"TruongCNN":<11} {df_cmp.auc.mean():>6.3f} {df_cmp.auc_pr.mean():>7.3f} '
      f'{df_cmp.sens_window.mean():>8.3f} {df_cmp.fpr_h_window.mean():>10.1f} '
      f'{df_cmp.sens_alarm.mean():>8.3f} {df_cmp.fpr_h_alarm.mean():>10.2f}')

print()
print('Reference points:')
print('  Truong 2018 patient-specific   Sens=0.81  FPR/h=0.16')
print('  Our V6b PDC SVM (LOPO)          Sens≈0.005 FPR/h≈0.3 (alarm-level)')
print('  Our V11 PDC LOSO patient-spec.  (see V11 output for comparison)')
print()
print('Outputs: lopo_v12_truong_alarm.csv, lopo_v12_truong_compare.csv')


Model          AUC  AUC-PR  Sens(w)   FPR/h(w)  Sens(a)   FPR/h(a)
----------------------------------------------------------------------
TruongCNN    0.516   0.487    0.419      143.7    0.000       0.00

Reference points:
  Truong 2018 patient-specific   Sens=0.81  FPR/h=0.16
  Our V6b PDC SVM (LOPO)          Sens≈0.005 FPR/h≈0.3 (alarm-level)
  Our V11 PDC LOSO patient-spec.  (see V11 output for comparison)

Outputs: lopo_v12_truong_alarm.csv, lopo_v12_truong_compare.csv


## 6 · Comparison: Truong methodology, two protocols

Headline interpretation for the thesis: if Truong's published Sens = 0.81
comes primarily from the spectrogram-CNN methodology, V12 LOPO should
also be high. If it comes primarily from the patient-specific protocol,
V12 LOPO should look like our other LOPO results.


In [7]:
# Cell 6 — Methodology vs protocol decomposition
def _load(p):
    if not os.path.exists(p): return None
    df = pd.read_csv(p)
    for c in ('patient', 'patient_id'):
        if c in df.columns: df = df[~df[c].astype(str).isin(['MEAN','STD'])]
    return df

methods = []
# Truong methodology
methods.append({'methodology': 'Truong spectrogram-CNN',
                'protocol': 'patient-specific (Truong 2018)',
                'auc': float('nan'), 'sens': 0.81, 'fpr_h': 0.16})
methods.append({'methodology': 'Truong spectrogram-CNN',
                'protocol': 'cross-patient LOPO (this notebook)',
                'auc': df_cmp.auc.mean(), 'sens': df_cmp.sens_alarm.mean(),
                'fpr_h': df_cmp.fpr_h_alarm.mean()})

# PDC methodology under each protocol (for triangulation)
for name in ('LR', 'SVM'):
    d = _load(os.path.join(RESULTS_DIR, f'lopo_v6b_alarm_{name}.csv'))
    if d is not None:
        methods.append({'methodology': f'PDC + {name}',
                        'protocol': 'cross-patient LOPO',
                        'auc': d['auc'].mean(), 'sens': d['sensitivity'].mean(),
                        'fpr_h': d['fpr_per_hour'].mean()})

cmp = pd.DataFrame(methods)
print(cmp.to_string(index=False))
cmp.to_csv(Path(RESULTS_DIR) / 'lopo_v12_methodology_vs_protocol.csv', index=False)


           methodology                           protocol      auc     sens   fpr_h
Truong spectrogram-CNN     patient-specific (Truong 2018)      NaN 0.810000 0.16000
Truong spectrogram-CNN cross-patient LOPO (this notebook) 0.515529 0.000000 0.00000
              PDC + LR                 cross-patient LOPO 0.570214 0.000929 0.07061
             PDC + SVM                 cross-patient LOPO 0.582595 0.000919 0.00560
